 # Validate data from API using Pydantic

#### Use this code snippet to get a random dad joke
import requests

headers = {"Accept": "application/json"}
response = requests.get("https://icanhazdadjoke.com/", headers=headers)

print(response.json())

a)

In [46]:
import requests

# we note that we want an answer in json format
headers = {"Accept": "application/json"}
response = requests.get("https://icanhazdadjoke.com/", headers=headers)
joke_data = response.json()
print(joke_data)

{'id': 'lyPZgVn3Le', 'joke': 'What did the ocean say to the shore? Nothing, it just waved.', 'status': 200}


b)

In [47]:
# create a pydantic model to validate the data

from pydantic import BaseModel, EmailStr, Field, ValidationError

# Create a pydantic model that matches the API response
class Joke(BaseModel):
    id: str
    joke: str
    status: int

# Validate the data from the API with our new model
try:
    validated_joke = Joke.model_validate(joke_data)
    print("Validation successful!")
    print(f"Joke ID: {validated_joke.id}")
    print(f"Joke: {validated_joke.joke}")
except ValidationError as e:
    print("Validation error! The data did not match the expected format.")
    print(e)

Validation successful!
Joke ID: lyPZgVn3Le
Joke: What did the ocean say to the shore? Nothing, it just waved.


c)


In [48]:
from pydantic import computed_field

# Create new joke model with a computed field

class Joke(BaseModel):
    id: str
    joke: str
    status: int

    # this is the new computed field
    @computed_field
    @property
    def words_in_joke(self) -> int:
        #split the joke into words and return the length of the list(number of words)
        return len(self.joke.split())

print("Model 'Joke' is now updated with a computed field 'words_in_joke'.")


Model 'Joke' is now updated with a computed field 'words_in_joke'.


In [49]:
# Validating the data from the API with our new model

# API-URL and neccessary headers to get a json response
API_URL = "https://icanhazdadjoke.com/"
HEADERS = {"Accept": "application/json"}

print("Fetching a random joke from the API...")

try:
    # Make the API request
    response = requests.get(API_URL, headers=HEADERS)
    response.raise_for_status() # Raise an error if the request was unsuccessful

    # Parse the JSON response
    joke_data = response.json()
    print("Joke data collected from API. Validating with Pydantic Joke model...")

    # Validate the data using the Joke model
    # **joke_data unpacks the dictionary into keyword arguments
    validated_joke = Joke(**joke_data)
    
    # Print the validated joke and the computed field
    print(f"Joke ID: {validated_joke.id}")
    print(f"Joke: {validated_joke.joke}")

    # Print the computed field
    print(f"Number of words in the joke: {validated_joke.words_in_joke}")
    
    print("--------------------------------------------------------------------")

    # The model as a dictionary
    print("Model as a dictionary:")
    print(validated_joke.model_dump())
except requests.exceptions.RequestException as req_err:
    print(f"Request error: {req_err}")



Fetching a random joke from the API...
Joke data collected from API. Validating with Pydantic Joke model...
Joke ID: xc21Lmbxcib
Joke: How did the hipster burn the roof of his mouth? He ate the pizza before it was cool.
Number of words in the joke: 18
--------------------------------------------------------------------
Model as a dictionary:
{'id': 'xc21Lmbxcib', 'joke': 'How did the hipster burn the roof of his mouth? He ate the pizza before it was cool.', 'status': 200, 'words_in_joke': 18}


d)

In [50]:
import time

# Use the same Joke-model as above
class Joke(BaseModel):
    id: str
    joke: str
    status: int

    @computed_field
    @property
    def words_in_joke(self) -> int:
        return len(self.joke.split())

# konstant logic to fetch and validate 10 jokes
API_URL = "https://icanhazdadjoke.com/"
HEADERS = {"Accept": "application/json"}
NUM_JOKES = 10
SLEEP_INTERVAL = 5 # seconds

# Create a empty list to store our validated jokes
validated_jokes_list = []

print(f"Fetching and validating {NUM_JOKES} jokes from the API...")


Fetching and validating 10 jokes from the API...


In [51]:
# Create a loop to fetch and validate multiple jokes
for i in range(NUM_JOKES):
    # write out the joke number we are fetching
    print(f"\n[{i+1}/{NUM_JOKES}] Getting a new joke...")
    
    try:
        # Make the API call
        response = requests.get(API_URL, headers=HEADERS)
        response.raise_for_status() # Raise an error if the request was unsuccessful

        # Validate the data using the Joke model
        joke_data = response.json()
        validated_joke = Joke(**joke_data)

        # Add the validated joke to our list
        validated_jokes_list.append(validated_joke)

        print(f"Joke retrieved and validated: {validated_joke.joke}")
    
    except requests.exceptions.RequestException as err:
        print(f"A problem occured while fetching the joke: #{i+1}. Fel: {err}")

    if i < NUM_JOKES - 1: # No need to sleep after the last joke
        print(f"Pausing for {SLEEP_INTERVAL} seconds before fetching the next joke...")
        time.sleep(SLEEP_INTERVAL)

# After the loop is done, print out the totalt number of validated jokes
print("----------------------------------------------------------------")
print(f"All {len(validated_jokes_list)} jokes have been fetched and validated.")
print("----------------------------------------------------------------")



[1/10] Getting a new joke...
Joke retrieved and validated: This morning I was wondering where the sun was, but then it dawned on me.
Pausing for 5 seconds before fetching the next joke...

[2/10] Getting a new joke...
Joke retrieved and validated: I accidentally drank a bottle of invisible ink. Now I’m in hospital, waiting to be seen.
Pausing for 5 seconds before fetching the next joke...

[3/10] Getting a new joke...
Joke retrieved and validated: People are making apocalypse jokes like there’s no tomorrow.
Pausing for 5 seconds before fetching the next joke...

[4/10] Getting a new joke...
Joke retrieved and validated: *Reversing the car* "Ah, this takes me back"
Pausing for 5 seconds before fetching the next joke...

[5/10] Getting a new joke...
Joke retrieved and validated: What did the green grape say to the purple grape?
BREATH!!
Pausing for 5 seconds before fetching the next joke...

[6/10] Getting a new joke...
Joke retrieved and validated: This morning I was wondering where th

In [53]:
for index, joke_obj in enumerate(validated_jokes_list, start=1):
    print(f"Joke: {index}:")
    print(f"   Text: {joke_obj.joke}")
    print(f"   Words in joke: {joke_obj.words_in_joke}")
    print("-"*20)



Joke: 1:
   Text: This morning I was wondering where the sun was, but then it dawned on me.
   Words in joke: 15
--------------------
Joke: 2:
   Text: I accidentally drank a bottle of invisible ink. Now I’m in hospital, waiting to be seen.
   Words in joke: 16
--------------------
Joke: 3:
   Text: People are making apocalypse jokes like there’s no tomorrow.
   Words in joke: 9
--------------------
Joke: 4:
   Text: *Reversing the car* "Ah, this takes me back"
   Words in joke: 8
--------------------
Joke: 5:
   Text: What did the green grape say to the purple grape?
BREATH!!
   Words in joke: 11
--------------------
Joke: 6:
   Text: This morning I was wondering where the sun was, but then it dawned on me.
   Words in joke: 15
--------------------
Joke: 7:
   Text: Frankenstein enters a bodybuilding competition and finds he has seriously misunderstood the objective.
   Words in joke: 13
--------------------
Joke: 8:
   Text: What's the worst part about being a cross-eyed teacher?

Th